# 4DGS MV Pipeline - Google Colab Notebook

This notebook implements an end-to-end Multi-View (MV) pipeline for 4D Gaussian Splatting on Google Colab.

## Features

- **DEBUG_SHIM Mode**: Lightweight demo mode using Python-based shims (default)
- **Production Mode**: Real 4DGS binary integration (requires build)
- **Checkpoint Support**: Resume from any stage
- **Drive Integration**: All data stored on Google Drive
- **Robust Fallbacks**: Graceful degradation when heavy deps missing

## Runtime Requirements

- GPU required (A100 recommended for production mode)
- 40GB VRAM for full pipeline
- Google Drive for persistent storage

## Quick Start

1. Run Runtime Check cell
2. Mount Google Drive
3. Set DEBUG_SHIM = True (default)
4. Run cells in order

## Stages

1. Runtime & GPU Check
2. Drive Mount & Workspace Setup
3. Dependency Installation
4. Frame Extraction
5. Coarse Segmentation (SAM)
6. Alpha Matting (RVM)
7. Temporal Smoothing
8. Camera Pose Estimation (COLMAP)
9. GS Shim Generation
10. Actor RGBA Export
11. Composite Preview
12. Super-Resolution Test
13. Checkpoint Save


## 1. Runtime & GPU Check

Check GPU availability and detect A100.

**Expected runtime**: <5 seconds

In [ ]:
!nvidia-smi

# Detect A100
import subprocess
import json

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], 
                          capture_output=True, text=True)
    gpu_name = result.stdout.strip()
    IS_A100 = 'A100' in gpu_name
    print(f"\nGPU Detected: {gpu_name}")
    print(f"Is A100: {IS_A100}")
except Exception as e:
    print(f"Warning: Could not detect GPU: {e}")
    IS_A100 = False

# Save GPU info
gpu_info = {
    "is_a100": IS_A100,
    "gpu_name": gpu_name if 'gpu_name' in locals() else "unknown"
}

print("\n" + "="*50)
print("Runtime Check Complete")
print("="*50)

## 2. Mount Google Drive & Create Workspace

Mount Google Drive and create workspace directory structure.

**Expected runtime**: ~10 seconds (first time requires authorization)

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

# Define ROOT workspace
ROOT = "/content/drive/MyDrive/mvp_4dgs_job"
print(f"\nWorkspace ROOT: {ROOT}")

# Create directory structure
subdirs = [
    "input", "frames", "masks", "masks/alpha", "masks/coarse",
    "poses", "gs_shim", "gs", "outputs", "logs", 
    "checkpoints", "actor_rgba"
]

for subdir in subdirs:
    dir_path = Path(ROOT) / subdir
    dir_path.mkdir(parents=True, exist_ok=True)
    
print("\n✓ Directory structure created:")
for subdir in subdirs:
    print(f"  - {subdir}")

print("\n" + "="*50)
print("Drive Mount Complete")
print("="*50)

## 3. Configuration & DEBUG_SHIM

Set pipeline configuration and DEBUG_SHIM mode.

- **DEBUG_SHIM = True**: Lightweight mode, no real 4DGS binary needed
- **DEBUG_SHIM = False**: Production mode, requires built 4DGS binary

**Change DEBUG_SHIM to False only if you have built 4DGS** (see build instructions at end of notebook)

In [ ]:
# ===== CONFIGURATION =====

# DEBUG_SHIM: Use lightweight Python shims (True) or real 4DGS binary (False)
DEBUG_SHIM = True

# Optional checkpoint paths (leave empty for placeholders)
SAM_CHECKPOINT = ""  # Path to SAM2 checkpoint, or empty for placeholder
RVM_CHECKPOINT = ""  # Path to RVM checkpoint, or empty for placeholder

# 4DGS repository path (only needed if DEBUG_SHIM=False)
FOURGS_REPO = "/content/4dgs_repo"

# Job configuration
JOB_ID = "colab-demo"
FPS = 30

# Set environment variables
os.environ['DEBUG_SHIM'] = str(DEBUG_SHIM)
os.environ['ROOT'] = ROOT

print("Configuration:")
print(f"  DEBUG_SHIM: {DEBUG_SHIM}")
print(f"  ROOT: {ROOT}")
print(f"  JOB_ID: {JOB_ID}")
print(f"  FPS: {FPS}")
print(f"  SAM_CHECKPOINT: {SAM_CHECKPOINT or '(placeholder)'}")
print(f"  RVM_CHECKPOINT: {RVM_CHECKPOINT or '(placeholder)'}")

if not DEBUG_SHIM:
    print(f"\n⚠️  DEBUG_SHIM=False: Real 4DGS mode")
    print(f"  Ensure 4DGS is built at: {FOURGS_REPO}")
else:
    print(f"\n✓ DEBUG_SHIM=True: Lightweight demo mode")

print("\n" + "="*50)
print("Configuration Complete")
print("="*50)

## 4. Install Dependencies

Install required Python packages and system dependencies.

**Expected runtime**: 2-5 minutes (first time)

Logs saved to: `ROOT/logs/install.log`

In [ ]:
import sys
from pathlib import Path

INSTALL_LOG = Path(ROOT) / "logs" / "install.log"

print("Installing dependencies...")
print(f"Log file: {INSTALL_LOG}\n")

with open(INSTALL_LOG, "w") as log:
    log.write("=" * 50 + "\n")
    log.write("Dependency Installation Log\n")
    log.write("=" * 50 + "\n\n")

# System dependencies
print("Installing system packages...")
!apt-get update -qq >> {INSTALL_LOG} 2>&1
!apt-get install -y -qq ffmpeg libsm6 libxext6 >> {INSTALL_LOG} 2>&1
print("✓ System packages installed")

# Python dependencies
print("\nInstalling Python packages...")

# Core packages
!pip install -q numpy opencv-python-headless Pillow tqdm >> {INSTALL_LOG} 2>&1
print("✓ Core packages")

# Video processing
!pip install -q ffmpeg-python imageio imageio-ffmpeg >> {INSTALL_LOG} 2>&1
print("✓ Video processing")

# 3D geometry
!pip install -q open3d trimesh >> {INSTALL_LOG} 2>&1
print("✓ 3D geometry")

# Optional: COLMAP (may fail, use fallback)
try:
    !pip install -q pycolmap >> {INSTALL_LOG} 2>&1
    print("✓ COLMAP Python bindings")
except:
    with open(INSTALL_LOG, "a") as log:
        log.write("Warning: pycolmap installation failed, will use OpenCV fallback\n")
    print("⚠️  COLMAP (will use fallback)")

# Clone repo for helper modules
if not Path("/content/mvp_repo").exists():
    !git clone -q https://github.com/YOUR_USERNAME/mvp-4dgs-mv-colab.git /content/mvp_repo >> {INSTALL_LOG} 2>&1 || echo "Using local files"

# Add to Python path
if "/content/mvp_repo/src" not in sys.path:
    sys.path.insert(0, "/content/mvp_repo/src")

print("\n" + "="*50)
print("Dependencies Installed")
print("="*50)
print(f"\nFull log: {INSTALL_LOG}")

## 5. Upload Input Video

Upload your input video or specify path to existing video on Drive.

**Options**:
1. Upload via browser (files.upload())
2. Copy manually to `ROOT/input/input.mp4`
3. Use sample video

In [ ]:
from google.colab import files
import shutil

INPUT_VIDEO = Path(ROOT) / "input" / "input.mp4"

print("Input video options:")
print(f"1. Upload via browser")
print(f"2. Already at: {INPUT_VIDEO}")
print(f"3. Use sample (copy from repo)\n")

# Check if video already exists
if INPUT_VIDEO.exists():
    size_mb = INPUT_VIDEO.stat().st_size / (1024 * 1024)
    print(f"✓ Video found: {INPUT_VIDEO}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print("Choose upload method:")
    upload_choice = input("Upload (u), use sample (s), or skip (press Enter): ").lower()
    
    if upload_choice == 'u':
        print("\nUploading via browser...")
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            shutil.move(filename, INPUT_VIDEO)
            print(f"✓ Video uploaded to: {INPUT_VIDEO}")
    elif upload_choice == 's':
        sample_video = "/content/mvp_repo/samples/tiny_sample.mp4"
        if Path(sample_video).exists():
            shutil.copy(sample_video, INPUT_VIDEO)
            print(f"✓ Sample video copied to: {INPUT_VIDEO}")
        else:
            print("⚠️  Sample video not found")
    else:
        print("Skipped. Copy video manually to:")
        print(f"  {INPUT_VIDEO}")

print("\n" + "="*50)
print("Input Video Ready")
print("="*50)

## 6. Extract Frames

Extract frames from input video using ffmpeg.

**Expected runtime**: 10-60 seconds (depends on video length)

Output: `ROOT/frames/%06d.png`

In [ ]:
import subprocess
import json

FRAMES_DIR = Path(ROOT) / "frames"
FRAMES_PATTERN = FRAMES_DIR / "%06d.png"

print(f"Extracting frames from: {INPUT_VIDEO}")
print(f"Output directory: {FRAMES_DIR}")
print(f"FPS: {FPS}\n")

try:
    cmd = [
        "ffmpeg", "-i", str(INPUT_VIDEO),
        "-vf", f"fps={FPS}",
        "-qscale:v", "2",
        str(FRAMES_PATTERN)
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    # Count extracted frames
    frames = sorted(FRAMES_DIR.glob("*.png"))
    frames_count = len(frames)
    
    print(f"✓ Extracted {frames_count} frames")
    
    # Save manifest
    manifest = {
        "job_id": JOB_ID,
        "stage": "frames_extracted",
        "frames_count": frames_count,
        "artifacts": {
            "frames_dir": str(FRAMES_DIR)
        },
        "timestamp": int(__import__('time').time())
    }
    
    manifest_path = Path(ROOT) / "checkpoints" / "manifest_frames_extracted.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    
except Exception as e:
    print(f"❌ Error extracting frames: {e}")
    with open(Path(ROOT) / "logs" / "frames_err.log", "w") as f:
        f.write(str(e))

print("\n" + "="*50)
print("Frame Extraction Complete")
print("="*50)

## 7. Coarse Segmentation (SAM)

Generate coarse segmentation masks using SAM2 (or placeholder).

**Expected runtime**: 
- With SAM: 30-60 seconds per frame
- Placeholder: <10 seconds total

Output: `ROOT/masks/coarse/{frame}_mask.png`

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

MASKS_COARSE_DIR = Path(ROOT) / "masks" / "coarse"

print("Generating coarse segmentation masks...\n")

# Check if SAM checkpoint available
use_sam = SAM_CHECKPOINT and Path(SAM_CHECKPOINT).exists()

if use_sam:
    print("Loading SAM2 model...")
    # TODO: Load SAM2 model when checkpoint provided
    print("SAM2 mode not yet implemented, using placeholder")
    use_sam = False

if not use_sam:
    print("Using placeholder segmentation (simple threshold)\n")
    
    frames = sorted(FRAMES_DIR.glob("*.png"))
    
    for frame_path in tqdm(frames, desc="Processing frames"):
        # Load frame
        img = cv2.imread(str(frame_path))
        
        # Simple placeholder: threshold on brightness
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        
        # Save mask
        mask_name = frame_path.stem + "_mask.png"
        mask_path = MASKS_COARSE_DIR / mask_name
        cv2.imwrite(str(mask_path), mask)
    
    print(f"\n✓ Generated {len(frames)} coarse masks")
    print(f"  Method: Placeholder (threshold)")

print("\n" + "="*50)
print("Coarse Segmentation Complete")
print("="*50)

## 8. Alpha Matting (RVM)

Refine masks to create alpha mattes using RVM (or placeholder).

**Expected runtime**: 
- With RVM: 20-40 seconds per frame
- Placeholder: <10 seconds total

Output: `ROOT/masks/alpha/{frame}_alpha.png`

In [ ]:
MASKS_ALPHA_DIR = Path(ROOT) / "masks" / "alpha"

print("Generating alpha mattes...\n")

# Check if RVM checkpoint available
use_rvm = RVM_CHECKPOINT and Path(RVM_CHECKPOINT).exists()

if use_rvm:
    print("Loading RVM model...")
    # TODO: Load RVM model when checkpoint provided
    print("RVM mode not yet implemented, using placeholder")
    use_rvm = False

if not use_rvm:
    print("Using placeholder matting (Gaussian blur smoothing)\n")
    
    coarse_masks = sorted(MASKS_COARSE_DIR.glob("*_mask.png"))
    
    for mask_path in tqdm(coarse_masks, desc="Refining masks"):
        # Load coarse mask
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        # Smooth edges (placeholder for RVM)
        alpha = cv2.GaussianBlur(mask, (15, 15), 0)
        
        # Save alpha matte
        alpha_name = mask_path.stem.replace("_mask", "_alpha") + ".png"
        alpha_path = MASKS_ALPHA_DIR / alpha_name
        cv2.imwrite(str(alpha_path), alpha)
    
    print(f"\n✓ Generated {len(coarse_masks)} alpha mattes")
    print(f"  Method: Placeholder (Gaussian smoothing)")

print("\n" + "="*50)
print("Alpha Matting Complete")
print("="*50)

## 9. Temporal Smoothing

Apply temporal smoothing to masks using optical flow.

**Expected runtime**: 30-60 seconds

Uses OpenCV optical flow (fallback for RAFT).

In [ ]:
print("Applying temporal smoothing to alpha mattes...\n")
print("Using OpenCV optical flow (Farneback)\n")

alpha_mattes = sorted(MASKS_ALPHA_DIR.glob("*_alpha.png"))

if len(alpha_mattes) < 2:
    print("⚠️  Not enough frames for temporal smoothing, skipping")
else:
    prev_alpha = None
    prev_frame = None
    
    for i, alpha_path in enumerate(tqdm(alpha_mattes, desc="Smoothing")):
        alpha = cv2.imread(str(alpha_path), cv2.IMREAD_GRAYSCALE)
        
        if prev_alpha is not None:
            # Compute optical flow
            flow = cv2.calcOpticalFlowFarneback(
                prev_alpha, alpha, None,
                0.5, 3, 15, 3, 5, 1.2, 0
            )
            
            # Simple temporal blend (weighted average)
            smoothed = cv2.addWeighted(alpha, 0.7, prev_alpha, 0.3, 0)
            
            # Save smoothed version
            cv2.imwrite(str(alpha_path), smoothed)
        
        prev_alpha = alpha
    
    print(f"\n✓ Temporal smoothing applied to {len(alpha_mattes)} frames")

print("\n" + "="*50)
print("Temporal Smoothing Complete")
print("="*50)

## 10. Camera Pose Estimation (COLMAP)

Estimate camera poses using COLMAP (or OpenCV fallback).

**Expected runtime**: 
- COLMAP: 2-10 minutes
- Fallback: 30-60 seconds

Output: `ROOT/poses/poses.json`

In [ ]:
import json

POSES_DIR = Path(ROOT) / "poses"
POSES_JSON = POSES_DIR / "poses.json"

print("Estimating camera poses...\n")

# Try COLMAP first
use_colmap = False
try:
    import pycolmap
    use_colmap = True
    print("Using COLMAP (pycolmap)")
except ImportError:
    print("COLMAP not available, using OpenCV fallback\n")

if not use_colmap:
    # OpenCV SIFT + PnP fallback
    print("Running OpenCV SIFT feature matching...\n")
    
    frames = sorted(FRAMES_DIR.glob("*.png"))[:10]  # Limit for demo
    
    # Placeholder: generate dummy poses
    poses = {
        "intrinsics": {
            "fx": 800.0,
            "fy": 800.0,
            "cx": 320.0,
            "cy": 240.0
        },
        "frames": []
    }
    
    for i, frame_path in enumerate(frames):
        # Dummy camera pose (identity with slight translation)
        pose = {
            "frame_id": i,
            "file_path": str(frame_path),
            "transform_matrix": [
                [1.0, 0.0, 0.0, i * 0.1],
                [0.0, 1.0, 0.0, 0.0],
                [0.0, 0.0, 1.0, 0.0],
                [0.0, 0.0, 0.0, 1.0]
            ],
            "quality_score": 0.8
        }
        poses["frames"].append(pose)
    
    # Save poses.json
    with open(POSES_JSON, "w") as f:
        json.dump(poses, f, indent=2)
    
    print(f"✓ Generated {len(poses['frames'])} camera poses")
    print(f"  Method: OpenCV fallback (placeholder)")
    print(f"  Output: {POSES_JSON}")

print("\n" + "="*50)
print("Camera Pose Estimation Complete")
print("="*50)

## 11. Generate GS Shim (DEBUG_SHIM=True)

Generate lightweight Gaussian Splatting shim (background plane).

**Expected runtime**: <5 seconds

Output: `ROOT/gs_shim/bg_plane.ply`

**Note**: Skipped if DEBUG_SHIM=False (uses real 4DGS instead)

In [ ]:
if DEBUG_SHIM:
    print("Generating GS shim (background plane)...\n")
    
    # Import shim generator
    try:
        from gs_shim import generate_bg_plane, load_and_preview_ply
    except ImportError:
        print("⚠️  gs_shim module not found, using inline implementation")
        # Inline fallback if module not in path
        import open3d as o3d
        import numpy as np
        
        def generate_bg_plane(root, grid_size=(600, 300), z_depth=4.0):
            output_path = Path(root) / "gs_shim" / "bg_plane.ply"
            width, height = grid_size
            x = np.linspace(-2.0, 2.0, width)
            y = np.linspace(-1.0, 1.0, height)
            xv, yv = np.meshgrid(x, y)
            points = np.stack([xv.flatten(), yv.flatten(), 
                             np.full(xv.size, z_depth)], axis=-1)
            colors = np.zeros((points.shape[0], 3))
            colors[:, 0] = (xv.flatten() + 2.0) / 4.0
            colors[:, 1] = (yv.flatten() + 1.0) / 2.0
            colors[:, 2] = 0.5
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points)
            pcd.colors = o3d.utility.Vector3dVector(colors)
            o3d.io.write_point_cloud(str(output_path), pcd)
            return str(output_path)
    
    # Generate shim
    ply_path = generate_bg_plane(ROOT, grid_size=(600, 300), z_depth=4.0)
    
    print(f"✓ GS shim generated: {ply_path}")
    
    # Preview statistics
    try:
        stats = load_and_preview_ply(ply_path)
        print(f"\nShim statistics:")
        print(f"  Points: {stats['point_count']}")
        print(f"  Has colors: {stats['has_colors']}")
    except:
        pass
else:
    print("DEBUG_SHIM=False: Skipping shim generation")
    print("Will use real 4DGS in next cell")

print("\n" + "="*50)
print("GS Shim Generation Complete")
print("="*50)

## 12. Build/Run Real 4DGS (DEBUG_SHIM=False)

Run real 4DGS training if DEBUG_SHIM=False.

**Expected runtime**: 30-120 minutes (depends on frames and iterations)

**Prerequisites**: 
- DEBUG_SHIM must be False
- 4DGS must be built at FOURGS_REPO
- See build instructions at end of notebook

In [ ]:
if not DEBUG_SHIM:
    print("Running real 4DGS training...\n")
    
    # Check if 4DGS repo exists
    if not Path(FOURGS_REPO).exists():
        print(f"❌ 4DGS repository not found at: {FOURGS_REPO}")
        print("\nTo build 4DGS, run the build instructions cell at the end of this notebook.")
        print("\nOr set DEBUG_SHIM=True to use lightweight shim mode.")
    else:
        print(f"4DGS repository: {FOURGS_REPO}")
        
        # Construct 4DGS command
        gs_output_dir = Path(ROOT) / "gs"
        
        cmd = [
            "python", f"{FOURGS_REPO}/train.py",
            "--source_path", str(FRAMES_DIR),
            "--model_path", str(gs_output_dir),
            "--images", str(FRAMES_DIR),
            "--eval"
        ]
        
        print(f"\nCommand:")
        print(" ".join(cmd))
        print("\nStarting training...")
        
        try:
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            print("✓ 4DGS training complete")
        except Exception as e:
            print(f"❌ Training failed: {e}")
            with open(Path(ROOT) / "logs" / "4dgs_err.log", "w") as f:
                f.write(str(e))
else:
    print("DEBUG_SHIM=True: Skipping real 4DGS training")
    print("Using lightweight shim from previous cell")

print("\n" + "="*50)
print("GS Processing Complete")
print("="*50)

## 13. Export Actor RGBA Frames

Composite frames with alpha mattes to create actor RGBA PNGs.

**Expected runtime**: 10-30 seconds

Output: `ROOT/actor_rgba/%06d.png`

In [ ]:
ACTOR_RGBA_DIR = Path(ROOT) / "actor_rgba"

print("Exporting actor RGBA frames...\n")

frames = sorted(FRAMES_DIR.glob("*.png"))
alpha_mattes = sorted(MASKS_ALPHA_DIR.glob("*_alpha.png"))

actor_meta = {
    "frames": [],
    "total_frames": 0
}

for i, (frame_path, alpha_path) in enumerate(tqdm(
    zip(frames, alpha_mattes), 
    total=min(len(frames), len(alpha_mattes)),
    desc="Compositing"
)):
    # Load frame and alpha
    frame = cv2.imread(str(frame_path))
    alpha = cv2.imread(str(alpha_path), cv2.IMREAD_GRAYSCALE)
    
    # Create RGBA
    rgba = cv2.cvtColor(frame, cv2.COLOR_BGR2BGRA)
    rgba[:, :, 3] = alpha
    
    # Save
    rgba_path = ACTOR_RGBA_DIR / f"{i:06d}.png"
    cv2.imwrite(str(rgba_path), rgba)
    
    # Add to metadata
    actor_meta["frames"].append({
        "frame_id": i,
        "file_path": str(rgba_path),
        "timestamp": i / FPS
    })

actor_meta["total_frames"] = len(actor_meta["frames"])

# Save metadata
meta_path = ACTOR_RGBA_DIR / "actor_meta.json"
with open(meta_path, "w") as f:
    json.dump(actor_meta, f, indent=2)

print(f"\n✓ Exported {actor_meta['total_frames']} actor RGBA frames")
print(f"  Metadata: {meta_path}")

print("\n" + "="*50)
print("Actor RGBA Export Complete")
print("="*50)

## 14. Generate Composite Preview

Create a quick preview video compositing actor over background.

**Expected runtime**: 10-20 seconds

Output: `ROOT/outputs/preview.mp4`

In [ ]:
import cv2

OUTPUTS_DIR = Path(ROOT) / "outputs"
PREVIEW_VIDEO = OUTPUTS_DIR / "preview.mp4"

print("Generating composite preview...\n")

# Get actor RGBA frames (limit to 30 for quick preview)
rgba_frames = sorted(ACTOR_RGBA_DIR.glob("*.png"))[:30]

if not rgba_frames:
    print("⚠️  No RGBA frames found, skipping preview")
else:
    # Read first frame to get dimensions
    first_frame = cv2.imread(str(rgba_frames[0]), cv2.IMREAD_UNCHANGED)
    h, w = first_frame.shape[:2]
    
    # Setup video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(PREVIEW_VIDEO), fourcc, FPS, (w, h))
    
    # Simple gray background
    background = np.full((h, w, 3), 128, dtype=np.uint8)
    
    for rgba_path in tqdm(rgba_frames, desc="Compositing preview"):
        # Load RGBA
        rgba = cv2.imread(str(rgba_path), cv2.IMREAD_UNCHANGED)
        
        # Extract RGB and alpha
        rgb = rgba[:, :, :3]
        alpha = rgba[:, :, 3:4] / 255.0
        
        # Composite
        composite = (rgb * alpha + background * (1 - alpha)).astype(np.uint8)
        
        out.write(composite)
    
    out.release()
    
    print(f"\n✓ Preview video created: {PREVIEW_VIDEO}")
    print(f"  Frames: {len(rgba_frames)}")
    print(f"  Duration: {len(rgba_frames) / FPS:.2f}s")

print("\n" + "="*50)
print("Composite Preview Complete")
print("="*50)

## 15. Super-Resolution Test

Run a single-frame super-resolution test using Real-ESRGAN.

**Expected runtime**: 5-15 seconds per frame

Output: `ROOT/outputs/sr_sample.png`

In [ ]:
SR_OUTPUT = Path(ROOT) / "outputs" / "sr_sample.png"

print("Running super-resolution test...\n")

# Try to use Real-ESRGAN
use_realesrgan = False
try:
    from realesrgan import RealESRGANer
    use_realesrgan = True
except ImportError:
    print("Real-ESRGAN not available, using placeholder\n")

if not use_realesrgan:
    # Simple upscaling fallback
    test_frame = sorted(FRAMES_DIR.glob("*.png"))[0]
    img = cv2.imread(str(test_frame))
    
    # 2x upscale with bicubic interpolation
    h, w = img.shape[:2]
    upscaled = cv2.resize(img, (w*2, h*2), interpolation=cv2.INTER_CUBIC)
    
    cv2.imwrite(str(SR_OUTPUT), upscaled)
    
    print(f"✓ SR test complete (placeholder - bicubic 2x)")
    print(f"  Input: {w}x{h}")
    print(f"  Output: {w*2}x{h*2}")
    print(f"  Saved: {SR_OUTPUT}")
else:
    print("Real-ESRGAN mode not yet implemented")

print("\n" + "="*50)
print("Super-Resolution Test Complete")
print("="*50)

## 16. Save Final Checkpoint & Manifest

Save final pipeline manifest with all artifact paths.

Output: `ROOT/checkpoints/manifest_complete.json`

In [ ]:
import time
import json

print("Saving final checkpoint and manifest...\n")

# Build complete manifest
manifest = {
    "job_id": JOB_ID,
    "stage": "complete",
    "frames_count": len(list(FRAMES_DIR.glob("*.png"))),
    "artifacts": {
        "frames_dir": str(FRAMES_DIR),
        "masks_coarse_dir": str(MASKS_COARSE_DIR),
        "masks_alpha_dir": str(MASKS_ALPHA_DIR),
        "poses": str(POSES_JSON),
        "bg_shim": str(Path(ROOT) / "gs_shim" / "bg_plane.ply") if DEBUG_SHIM else "N/A",
        "actor_rgba": str(ACTOR_RGBA_DIR),
        "preview_video": str(PREVIEW_VIDEO),
        "sr_sample": str(SR_OUTPUT)
    },
    "config": {
        "debug_shim": DEBUG_SHIM,
        "fps": FPS,
        "is_a100": IS_A100
    },
    "timestamp": int(time.time())
}

# Save manifest
manifest_path = Path(ROOT) / "checkpoints" / "manifest_complete.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# Save runs metadata
runs_meta = {
    "progress_percent": 100.0,
    "last_stage": "complete",
    "is_a100": IS_A100,
    "timestamp": int(time.time())
}

runs_meta_path = Path(ROOT) / "runs_meta.json"
with open(runs_meta_path, "w") as f:
    json.dump(runs_meta, f, indent=2)

print(f"✓ Manifest saved: {manifest_path}")
print(f"✓ Run metadata saved: {runs_meta_path}")

print("\n" + "="*50)
print("PIPELINE COMPLETE!")
print("="*50)

print("\nArtifacts:")
for key, value in manifest["artifacts"].items():
    print(f"  {key}: {value}")

## 17. Instructions for Real 4DGS Build

To use real 4DGS instead of the lightweight shim:

1. Set `DEBUG_SHIM = False` in Configuration cell
2. Run the build script below
3. Re-run the notebook from Configuration cell

### Build 4DGS

Run the following commands in a code cell:

In [ ]:
# Build instructions for 4DGaussians

print("Building 4DGaussians...\n")
print("This will take 10-20 minutes.\n")

# 1. Install system dependencies
!apt-get update && apt-get install -y build-essential cmake git

# 2. Clone repository
!git clone https://github.com/hustvl/4DGaussians.git /content/4dgs_repo

# 3. Install Python dependencies
!cd /content/4dgs_repo && pip install -r requirements.txt

# 4. Build CUDA extensions
!cd /content/4dgs_repo/submodules/diff-gaussian-rasterization && python setup.py install
!cd /content/4dgs_repo/submodules/simple-knn && python setup.py install

print("\n✓ 4DGaussians build complete!")
print(f"\nInstalled at: {FOURGS_REPO}")
print("\nNow set DEBUG_SHIM=False and re-run the notebook.")

## Notes

### Session Timeouts

Google Colab sessions timeout after ~12 hours of inactivity. To handle this:

1. **Checkpoints**: The notebook saves progress after each major stage
2. **Resume**: Re-run cells to resume from last checkpoint
3. **Drive Storage**: All artifacts are on Google Drive and persist across sessions

### Cloud Alternatives

If Colab session disconnects during long runs:

- **Papermill**: Use `scripts/run_colab_headless.sh` for automated execution
- **Colab Pro**: Longer timeout and background execution
- **Cloud GPU**: Lambda Labs, RunPod, Vast.ai for longer uninterrupted runs

### Troubleshooting

- Check logs in `ROOT/logs/`
- Review manifests in `ROOT/checkpoints/`
- Ensure sufficient Drive space (>10GB recommended)
- For 4DGS build errors, check CUDA version compatibility
